# TourGuide — prototyp nowego pipeline'u (etap 3a)

Obecny przebieg: **GPS → Nominatim → LocationContext → WikimediaWorker → TavilyResearchWorker → wspólny store**. DDGS, równoległe uruchomienie i narrator zostaną dodane w następnych etapach. Kod wykonawczy znajduje się w modułach `AI/`; notebook służy do uruchomienia i obejrzenia wyników.

Przed uruchomieniem zainstaluj zależności z `AI/requirements.txt`. Klucze API wpisz w pierwszej komórce poniżej. Notebook używa identyfikującego `User-Agent` z adresem repozytorium; możesz go zmienić w tej samej komórce. Publicznego Nominatim używaj do pojedynczych ręcznych prób, bez seryjnego odświeżania komórki. Dane lokalizacyjne: © OpenStreetMap contributors (ODbL).

In [1]:
import sys
import time
from pathlib import Path

### api keys ###
RESEARCH_MODEL_NAME = 'z-ai/glm-5.3-flash'

NOMINATIM_USER_AGENT = 'AITourGuidePrototype/0.1 (https://github.com/GigaChady/AI_Tour_Guide)'
WIKIMEDIA_USER_AGENT = NOMINATIM_USER_AGENT

def find_ai_root():
    for candidate in (Path.cwd(), Path.cwd() / 'AI', Path.cwd().parent):
        if (candidate / 'narration' / 'knowledge').is_dir():
            return candidate.resolve()
    raise RuntimeError('Uruchom notebook z katalogu repozytorium, AI lub AI/experiments')

AI_ROOT = find_ai_root()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

LATITUDE = 51.1216
LONGITUDE = 17.0352
print('AI root:', AI_ROOT)
print('GPS:', LATITUDE, LONGITUDE)

AI root: C:\Documents\Kody\ZPI\repo\repov2\AI_Tour_Guide\AI
GPS: 51.1216 17.0352


In [2]:
from integrations.geocoding.nominatim.nominatim_client import NominatimClient
from integrations.knowledge.wikimedia.wikimedia_article_provider import WikimediaArticleProvider
from narration.knowledge import InMemoryKnowledgeStore
from narration.location_discovery.location_context_task import LocationContextTask
from narration.research import WikimediaWorker, WikimediaWorkerConfig

nominatim_client = NominatimClient(user_agent=NOMINATIM_USER_AGENT)
gps_key = (LATITUDE, LONGITUDE)
if globals().get('_cached_gps') == gps_key:
    location = _cached_location
    geocoding_latency_s = 0.0
else:
    wait_s = 1.0 - (time.monotonic() - globals().get('_last_nominatim_call', 0.0))
    if wait_s > 0:
        time.sleep(wait_s)
    geo_started = time.perf_counter()
    _last_nominatim_call = time.monotonic()
    location = LocationContextTask(nominatim_client).run(LATITUDE, LONGITUDE)
    geocoding_latency_s = time.perf_counter() - geo_started
    if any((location.neighbourhood, location.district, location.city,
            location.region, location.country)):
        _cached_gps, _cached_location = gps_key, location
print('LocationContext:', location.model_dump())
print(f'Reverse geocoding: {geocoding_latency_s:.2f} s (0.00 oznacza cache)')
if not any((location.neighbourhood, location.district, location.city,
            location.region, location.country)):
    raise RuntimeError('Nominatim nie zwrócił poziomów lokalizacji; sprawdź adres i User-Agent')

C:\miniconda\envs\zpi\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


LocationContext: {'lat': 51.1216, 'lon': 17.0352, 'street': 'Bolesława Chrobrego', 'neighbourhood': 'Nadodrze', 'district': '', 'city': 'Wrocław', 'region': 'województwo dolnośląskie', 'country': 'Polska'}
Reverse geocoding: 0.10 s (0.00 oznacza cache)


In [3]:
store = InMemoryKnowledgeStore()
article_provider = WikimediaArticleProvider(
    user_agent=WIKIMEDIA_USER_AGENT, languages=('pl', 'en'),
    max_requests=12, timeout_seconds=8,
)
worker = WikimediaWorker(
    article_provider, store,
    WikimediaWorkerConfig(max_articles=3, fetch_sections_for_first_n_articles=2),
)
worker_started = time.perf_counter()
def show_chunk(chunk):
    elapsed = time.perf_counter() - worker_started
    print(f'+{elapsed:.2f}s [{chunk.scope.value}] {chunk.anchor_name} / {chunk.topic}: '
          f'{chunk.content[:120]}...')

run = worker.run(location, on_chunk=show_chunk)
print(f'Pierwszy chunk: {run.first_chunk_latency_s!r} s')
print(f'Cały WikimediaWorker: {run.latency_s:.2f} s')
print(f'Żądania API: {run.api_request_count}, artykuły: {run.article_count}, '
      f'chunki: {len(run.chunk_ids)}')

+0.75s [DISTRICT] Nadodrze / overview: Nadodrze – osiedle we Wrocławiu, wydzielone w 1991 z większego wrocławskiego osiedla – Ołbina – leżącego teraz od niego ...
+1.15s [CITY] Wrocław / overview: Wrocław (łac. Vratislavia, niem. Breslau, cz. Vratislav) – miasto na prawach powiatu w południowo-zachodniej Polsce, sie...
+1.55s [CITY] Wrocław / history: Wrocław jest jednym z najstarszych miast Polski pod względem lokacji na prawach miejskich. W czasach antycznych w miejsc...
+1.55s [CITY] Wrocław / nazwa_miasta: Pochodzenie nazwy miasta nie jest jasne, ponieważ w najstarszych określeniach tej nazwy zapisanych przez Thietmara jako ...
+1.78s [REGION] województwo dolnośląskie / overview: Województwo dolnośląskie – jednostka podziału administracyjnego Polski, jedno z 16 województw, ze stolicą we Wrocławiu. ...
Pierwszy chunk: 0.7547386000005645 s
Cały WikimediaWorker: 1.78 s
Żądania API: 7, artykuły: 3, chunki: 5


In [4]:
print('Próby wyszukiwania:')
for attempt in run.attempts:
    print(attempt.scope.value, attempt.anchor_name, attempt.status,
          attempt.article_title, attempt.error)
print('\nŹródła:')
for source in store.sources:
    print(source.source_id, source.title, source.url)
print('\nChunki:')
for chunk in store.chunks:
    print(chunk.scope.value, chunk.anchor_name, chunk.topic,
          chunk.source_ids, chunk.content[:300])
assert all(chunk.source_ids for chunk in store.chunks)

Próby wyszukiwania:
DISTRICT Nadodrze found Nadodrze (Wrocław) 
CITY Wrocław found Wrocław 
REGION województwo dolnośląskie found Województwo dolnośląskie 

Źródła:
wikimedia:bf97af3fa2da5a306f94 Nadodrze (Wrocław) https://pl.wikipedia.org/wiki/Nadodrze_(Wroc%C5%82aw)
wikimedia:6247cec8a625a7113f40 Wrocław https://pl.wikipedia.org/wiki/Wroc%C5%82aw
wikimedia:0d048eb2ab394ad62046 Województwo dolnośląskie https://pl.wikipedia.org/wiki/Wojew%C3%B3dztwo_dolno%C5%9Bl%C4%85skie

Chunki:
DISTRICT Nadodrze overview ['wikimedia:bf97af3fa2da5a306f94'] Nadodrze – osiedle we Wrocławiu, wydzielone w 1991 z większego wrocławskiego osiedla – Ołbina – leżącego teraz od niego na wschód; graniczy też (na północy) z Kleczkowem, na zachodzie z lokowanym w średniowieczu Starym Miastem. Nadodrze, leżące na terenach nazywanych „Przedmieściem Odrzańskim” (niem
CITY Wrocław overview ['wikimedia:6247cec8a625a7113f40'] Wrocław (łac. Vratislavia, niem. Breslau, cz. Vratislav) – miasto na prawach powiatu w południ

## TavilyResearchWorker — lokalne źródła

Worker wyszukuje strony, szereguje je według trafności lokalnej i zbiera do dwóch udanych ekstrakcji (maksymalnie cztery próby). Połączony tekst trafia do modelu w jednym wywołaniu. Model tworzy samodzielne, parafrazowane chunki bez dosłownych cytatów. Każdy chunk zachowuje listę URL-i stron użytych w tym wsadzie; lista nie oznacza, że każdy URL potwierdza każdy fakt.

Limity wyszukiwania, ekstrakcji, długości chunków i wag rankingu są w `narration/research/config/web_research_config.py`; parametry modelu w `research_model_config.py`, a limity adaptera w `integrations/knowledge/tavily/config/tavily_config.py`.

Dla PLACE worker ustala współrzędne przez Nominatim na podstawie nazwy i miasta, a pasujące lokalizacje zapisuje w `place.locations`. Brak współrzędnych jest widoczny w diagnostyce i nie tworzy chunka PLACE.

Komórka poniżej pokazuje długość i brzegi surowych tekstów Tavily, tekstów po prostym czyszczeniu linków Markdown, tekstu wysłanego modelowi oraz pełną odpowiedź modelu.


In [5]:
from integrations.knowledge.tavily import TavilyResearchClient
from integrations.knowledge.tavily.markdown_cleanup import clean_extracted_markdown
from integrations.llm.nvidia.config import NvidiaConfig
from integrations.llm.nvidia.nvidia_language_model import NvidiaLanguageModel
from narration.knowledge import PlaceLocationResolver
from narration.research import TavilyResearchWorker, WebResearchConfig
from narration.research.config import ResearchModelConfig

if not NVIDIA_API_KEY.strip():
    raise RuntimeError('Wpisz NVIDIA_API_KEY w pierwszej komórce notebooka')
web_config = WebResearchConfig()
model_config = ResearchModelConfig()
research_model = NvidiaLanguageModel(NvidiaConfig(
    model_name=RESEARCH_MODEL_NAME, temperature=model_config.temperature,
    max_tokens=model_config.max_tokens, reasoning_effort=model_config.reasoning_effort,
    request_timeout_seconds=model_config.request_timeout_seconds,
    max_retries=model_config.max_retries,
), api_key=NVIDIA_API_KEY.strip())

def show_text_edges(label, value, limit=model_config.diagnostic_preview_chars):
    print(f'{label}: {len(value)} znaków')
    print(f'  początek: {value[:limit]!r}')
    print(f'  koniec: {value[-limit:]!r}')

def show_tavily_raw(url, raw_content, max_chars):
    print(f'\n[TAVILY EXTRACT] {url}')
    show_text_edges('Surowy tekst Tavily', raw_content)
    cleaned = clean_extracted_markdown(raw_content)
    show_text_edges('Po prostym czyszczeniu Markdown', cleaned)
    print(f'  po limicie: {min(len(cleaned), max_chars)} znaków')

class DebugResearchModel:
    def __init__(self, delegate):
        self.delegate = delegate

    def generate(self, messages):
        user_message = next(content for role, content in reversed(messages) if role == 'user')
        document_text = user_message.partition('RESEARCH TEXT:\n')[2]
        print('\n[MODEL INPUT] połączony tekst źródłowy')
        show_text_edges('Tekst wsadu przekazany modelowi', document_text)
        response = self.delegate.generate(messages)
        print(f'[MODEL RESPONSE] {len(response)} znaków')
        print(response)
        return response

tavily_client = TavilyResearchClient(
    api_key=TAVILY_API_KEY.strip(), max_requests=web_config.max_provider_requests,
    on_extract_raw_content=show_tavily_raw,
)
tavily_worker = TavilyResearchWorker(
    tavily_client, DebugResearchModel(research_model), store,
    web_config,
    place_locator=PlaceLocationResolver(nominatim_client, article_provider),
)
tavily_started = time.perf_counter()
def show_tavily_chunk(chunk):
    elapsed = time.perf_counter() - tavily_started
    print(f'+{elapsed:.2f}s [{chunk.scope.value}] {chunk.anchor_name} / {chunk.topic}: '
          f'{chunk.content[:model_config.chunk_preview_chars]}...')

tavily_run = tavily_worker.run(location, on_chunk=show_tavily_chunk)
print('Tavily queries:', *tavily_run.queries, sep='\n  ')
print('Tavily ranking (priorytet, URL):',
      *(f'{hit.score}: {hit.url}' for hit in tavily_run.ranked_hits), sep='\n  ')
print('Tavily read:', *tavily_run.extracted_urls, sep='\n  ')
print(f'Próby ekstrakcji: {tavily_run.extract_calls}; udane teksty: {len(tavily_run.extracted_urls)}')
print('Tekst wsadu (znaki według URL):', tavily_run.input_chars_by_url)
print(f'Tavily: {tavily_run.latency_s:.2f}s; API {tavily_run.api_request_count}; '
      f'LLM {tavily_run.llm_calls}; nowe chunki {tavily_run.new_chunk_count}; '
      f'miejsca {len(tavily_run.place_ids)}')
print('Miejsca bez potwierdzonych wspolrzednych:', tavily_run.unresolved_places)
if tavily_run.errors:
    print('Diagnostyka:', *tavily_run.errors, sep='\n  ')

C:\miniconda\envs\zpi\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found z-ai/glm-5.3-flash in available_models, but type is unknown and inference may fail.
  warnings.warn(



[TAVILY EXTRACT] https://www.hamiltonmay.pl/miejski-przewodnik/nadodrze
Surowy tekst Tavily: 10237 znaków
  początek: 'Skontaktuj się z nami Warszawa [(+48) 22 428 16 15](tel:48224281615) Kraków [(+48) 12 426 51 26](tel:48124265126) Wrocław [(+48) 71 727 19 76](tel:48717271976)\n\n1. [Hamilton May](/)\n2. Przewodnik\n\n## Nadodrze, Wrocław\n\n[Ogólne informacje \ufeff](#ogólne-informacje\n\n\ufeff) [Architektura i nieruchomości](#architektura-i-nieruchomości) [Tereny zielone i rekreacyjne](#tereny-zielone-i-rekreacyjne) [Połączenia transportowe](#połączenia-transportowe) [Lokalne udogodnienia \ufeff](#lokalne-udogodnienia\n\n\ufeff)\n\n### Rozkwit Nadodrza to efekt udanej rewitalizacji ostatniej dekady\n\nNowoczesne Nadodrze, dzięki swojemu niekonwencjonalnemu charakterowi, oczarowało młodych, zamożnych profesjonalistów.\n\nNadodrze  \n W Wrocław, Polska\n\nNadodrze  \n W Wrocław, Polska\n\n### Ogólne informacje \ufeff\n\nNadodrze przez lata uchodziło za dość surowe. W ciągu ostatniej

In [6]:
print('Miejsca Tavily:')
for place in store.places:
    if place.place_id in tavily_run.place_ids:
        print(place.canonical_name, 'aliasy:', [a.name for a in place.aliases],
              'ulica:', place.street, 'koordynaty:', (place.lat, place.lon),
              'pochodzenie punktu reprezentacyjnego:', place.coordinate_source)
        for point in place.locations:
            print('  punkt:', (point.lat, point.lon), 'nazwa:', point.name,
                  'pochodzenie:', point.coordinate_source, 'ID:', point.external_ids)
print('\nChunki Tavily i URL-e wsadu:')
source_by_id = {source.source_id: source for source in store.sources}
for chunk in store.chunks:
    if 'tavily' in chunk.source_workers:
        print(chunk.scope.value, chunk.anchor_name, chunk.topic, chunk.content)
        print('  URL-e wsadu:', chunk.source_urls)
        for source_id in chunk.source_ids:
            source = source_by_id[source_id]
            print('  źródło:', source.title, source.url)
            print('  fragment źródłowy:', source.evidence_text[:1100])
assert all(chunk.source_ids for chunk in store.chunks)

Miejsca Tavily:
Plac św. Macieja aliasy: ['Plac Świętego Macieja'] ulica: Plac Świętego Macieja koordynaty: (51.1205777, 17.0358606) pochodzenie punktu reprezentacyjnego: nominatim:way:23806588
  punkt: (51.1196799, 17.0360601) nazwa: plac Świętego Macieja pochodzenie: nominatim:way:447657529 ID: {'osm': 'way:447657529'}
  punkt: (51.1199138, 17.0353313) nazwa: Plac Świętego Macieja pochodzenie: nominatim:way:22879757 ID: {'osm': 'way:22879757'}
  punkt: (51.1205777, 17.0358606) nazwa: Plac Świętego Macieja pochodzenie: nominatim:way:23806588 ID: {'osm': 'way:23806588'}

Chunki Tavily i URL-e wsadu:
DISTRICT Nadodrze District overview and history Nadodrze is a Wrocław district carved out of Ołbin in 1991, lying across the Oder from the Old Town on land once called 'Oder Vorstadt' (Przedmieście Odrzańskie), incorporated into the city in 1808 after the fortifications were demolished. Lightly damaged in WWII, it became the 'temporary city centre' after the war: in April 1945 the Polish ad